# Single WAV Inference

给定一个 `.wav/.flac` 音频和训练 checkpoint，加载 Spanish SenseVoiceSmall CTC-attention 模型并打印 greedy CTC 推理结果。

In [ ]:
from pathlib import Path
import sys

import torch

# notebook 位于 spanish_sensevoice_asr/notebooks/，这里自动把 workspace root 加入 sys.path。
WORKSPACE_ROOT = Path.cwd()
if WORKSPACE_ROOT.name == "notebooks":
    WORKSPACE_ROOT = WORKSPACE_ROOT.parents[1]
elif WORKSPACE_ROOT.name == "spanish_sensevoice_asr":
    WORKSPACE_ROOT = WORKSPACE_ROOT.parent

if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

print(WORKSPACE_ROOT)

In [ ]:
from spanish_sensevoice_asr.model.ctc_model import SenseVoiceSmallCTC
from spanish_sensevoice_asr.model.features import load_fbank
from spanish_sensevoice_asr.model.tokenizer import SpanishAsrTokenizer

In [ ]:
# 改这里：权重路径和音频路径。
CHECKPOINT_PATH = WORKSPACE_ROOT / "spanish_sensevoice_asr/exp/es_unigram_1000_ctc_attention_lfr_dynamic_2gpu/best.pt"
AUDIO_PATH = WORKSPACE_ROOT / "sample.wav"

TOKENS_PATH = WORKSPACE_ROOT / "spanish_sensevoice_asr/data/vocab/es_unigram_1000/tokens.json"
SP_MODEL_PATH = WORKSPACE_ROOT / "spanish_sensevoice_asr/data/vocab/es_unigram_1000/es_unigram_1000.model"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE = 8000
NUM_MEL_BINS = 80

print("checkpoint:", CHECKPOINT_PATH)
print("audio:", AUDIO_PATH)
print("device:", DEVICE)

assert CHECKPOINT_PATH.is_file(), f"checkpoint not found: {CHECKPOINT_PATH}"
assert AUDIO_PATH.is_file(), f"audio not found: {AUDIO_PATH}"
assert TOKENS_PATH.is_file(), f"tokens not found: {TOKENS_PATH}"
assert SP_MODEL_PATH.is_file(), f"sp model not found: {SP_MODEL_PATH}"

In [ ]:
device = torch.device(DEVICE)
tokenizer = SpanishAsrTokenizer(TOKENS_PATH, SP_MODEL_PATH)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model_conf = checkpoint.get("model_conf", {})

lfr_m = int(model_conf.get("lfr_m", 7))
lfr_n = int(model_conf.get("lfr_n", 6))
input_dim = int(model_conf.get("input_dim", NUM_MEL_BINS * lfr_m))

print("vocab_size:", tokenizer.vocab_size)
print("checkpoint epoch:", checkpoint.get("epoch"))
print("checkpoint global_step:", checkpoint.get("global_step"))
print("checkpoint train_loss:", checkpoint.get("train_loss"))
print("checkpoint dev_loss:", checkpoint.get("dev_loss"))
print("lfr_m/lfr_n/input_dim:", lfr_m, lfr_n, input_dim)
print("model_conf:")
for key in sorted(model_conf):
    print(f"  {key}: {model_conf[key]}")

In [ ]:
model = SenseVoiceSmallCTC(
    vocab_size=tokenizer.vocab_size,
    input_dim=input_dim,
    encoder_dim=model_conf.get("encoder_dim", 512),
    num_layers=model_conf.get("num_layers", 50),
    num_heads=model_conf.get("num_heads", 4),
    feedforward_dim=model_conf.get("feedforward_dim", 2048),
    dropout=model_conf.get("dropout", 0.1),
    tp_blocks=model_conf.get("tp_blocks", 20),
    attention_dropout_rate=model_conf.get("attention_dropout_rate", 0.1),
    kernel_size=model_conf.get("kernel_size", 11),
    sanm_shfit=model_conf.get("sanm_shfit", 0),
    blank_id=model_conf.get("blank_id", tokenizer.blank_id),
    ignore_id=model_conf.get("ignore_id", tokenizer.ignore_id),
    sos=model_conf.get("sos", 1),
    eos=model_conf.get("eos", 2),
    length_normalized_loss=model_conf.get("length_normalized_loss", True),
    decoder_layers=model_conf.get("decoder_layers", 2),
    decoder_loss_weight=model_conf.get("decoder_loss_weight", 1.0),
    ctc_loss_weight=model_conf.get("ctc_loss_weight", 1.0),
).to(device)

missing, unexpected = model.load_state_dict(checkpoint["model"], strict=False)
model.eval()

print("missing keys:", missing)
print("unexpected keys:", unexpected)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
features = load_fbank(
    AUDIO_PATH,
    sample_rate=SAMPLE_RATE,
    num_mel_bins=NUM_MEL_BINS,
    cache_resampled_wav=True,
    lfr_m=lfr_m,
    lfr_n=lfr_n,
)
lengths = torch.tensor([features.size(0)], dtype=torch.long, device=device)
features = features.unsqueeze(0).to(device)

print("feature shape:", tuple(features.shape))
print("feature length:", int(lengths[0].item()))

In [ ]:
with torch.inference_mode():
    token_ids, output_lengths = model.greedy_decode(features, lengths)

raw_ids = token_ids[0, : int(output_lengths[0].item())].detach().cpu().tolist()
text = tokenizer.decode_ctc(raw_ids)

print("output length:", int(output_lengths[0].item()))
print("raw token ids head:", raw_ids[:80])
print()
print("ASR result:")
print(text)